# hmda-analyzer — HMDA Mortgage Lending Disparity Analyzer
## Demo: Analyzing Racial Disparities in Mortgage Lending

This notebook demonstrates how to use hmda-analyzer to:
- Load HMDA LAR data from the CFPB Data Browser API or sample data
- Compute denial rates by race and ethnicity
- Calculate disparity ratios vs White applicants
- Analyze denial rates by income band
- Map lending activity by census tract and identify lending deserts
- Benchmark individual lenders against the market
- Generate a full fair lending disparity report

Data source: CFPB HMDA Data Browser API (free, no API key required)
Coverage: 4,908 institutions, millions of loan applications (2024 data)


In [ ]:
import sys
sys.path.insert(0, '..')

from hmdaanalyzer import (
    load_sample, load_from_api,
    denial_rate_by_race, disparity_ratio,
    denial_rate_by_income_band, denial_reasons_by_race,
    lending_by_tract, lending_by_county, lending_by_state,
    lending_desert_score,
    lender_summary, lender_vs_market, top_lenders_by_volume,
    generate_disparity_report, summary_table,
)
from hmdaanalyzer.data.schema import ACTION_TAKEN, RACE_CODES, DENIAL_REASONS
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print("hmda-analyzer loaded successfully")
print(f"\nAction codes tracked: {len(ACTION_TAKEN)}")
print(f"Race codes tracked:   {len(RACE_CODES)}")
print(f"Denial reasons:       {len(DENIAL_REASONS)}")


## 1. Load HMDA Data

We use synthetic sample data here — realistic distributions based on
2023 national HMDA statistics. Replace with real API data to analyze
actual lending patterns.


In [ ]:
# Load synthetic sample data (no API required)
df = load_sample(n=10_000, seed=42)

# To load real CFPB data:
# df = load_from_api(year=2023, state="IL")         # Illinois only
# df = load_from_api(year=2023, lei="BANK_LEI_HERE") # Single lender

print(f"Total records:        {len(df):,}")
print(f"Columns:              {list(df.columns)}")
print(f"\nAction Taken Distribution:")
print(df["action_taken"].value_counts().sort_index().to_string())
print(f"\nRace Distribution:")
print(df["derived_race"].value_counts().to_string())


## 2. Overall Denial Rates

First look at overall approval and denial rates across the dataset.


In [ ]:
actionable = df[df["action_taken"].isin([1, 2, 3])]

total = len(actionable)
approved = actionable["is_approved"].sum()
denied = actionable["is_denied"].sum()

print(f"Actionable Applications: {total:,}")
print(f"Approved:                {approved:,} ({approved/total*100:.1f}%)")
print(f"Denied:                  {denied:,} ({denied/total*100:.1f}%)")
print(f"\nAverage Loan Amount:     ${actionable['loan_amount'].mean():,.0f}k")
print(f"Median Applicant Income: ${actionable['income'].median():,.0f}k")


## 3. Denial Rates by Race

The core of HMDA fair lending analysis — comparing denial rates
across racial and ethnic groups.


In [ ]:
rates = denial_rate_by_race(df)
print("Denial Rates by Race/Ethnicity:")
print("=" * 60)
for _, row in rates.iterrows():
    bar = "█" * int(row["denial_rate"] * 50)
    print(f"  {row['derived_race']:<40} {row['denial_rate']*100:5.1f}%  {bar}")
print(f"\n{rates.to_string(index=False)}")


## 4. Disparity Ratios

Disparity ratio = group denial rate / White applicant denial rate.

CFPB thresholds:
- **>= 2.0x** — HIGH disparity (triggers regulatory scrutiny)
- **>= 1.5x** — MODERATE disparity
- **< 1.5x** — LOW disparity


In [ ]:
disparities = disparity_ratio(df)

print("Disparity Ratios vs White Applicants:")
print("=" * 70)
for _, row in disparities.iterrows():
    if row["derived_race"] == "White":
        continue
    ratio = row["disparity_ratio"]
    level = row["disparity_level"]
    emoji = {"HIGH": "🔴", "MODERATE": "🟡", "LOW": "🟢", "FAVORABLE": "✅"}.get(level, "")
    print(f"  {row['derived_race']:<40} {ratio:5.2f}x  {emoji} {level}")

print(f"\n{disparities[['derived_race', 'denial_rate', 'disparity_ratio', 'disparity_level']].to_string(index=False)}")


## 5. Denial Rates by Income Band

Are low-income applicants denied at higher rates regardless of race?


In [ ]:
income_df = denial_rate_by_income_band(df)
print("Denial Rates by Income Band:")
print("=" * 50)
for _, row in income_df.iterrows():
    bar = "█" * int(row["denial_rate"] * 50)
    print(f"  {str(row['income_band']):<12} {row['denial_rate']*100:5.1f}%  {bar}")
print(f"\n{income_df.to_string(index=False)}")


## 6. Denial Reasons by Race

What reasons are cited for denying applications from different racial groups?


In [ ]:
reasons = denial_reasons_by_race(df)
print("Top Denial Reasons by Race:")
print("=" * 70)
for race in ["Black or African American", "Hispanic or Latino", "White"]:
    race_reasons = reasons[reasons["derived_race"] == race].head(3)
    if len(race_reasons) > 0:
        print(f"\n{race}:")
        for _, row in race_reasons.iterrows():
            print(f"  {row['denial_reason_label']:<45} {row['pct']:.1f}%")


## 7. Geographic Analysis — Lending by State

In [ ]:
state_df = lending_by_state(df)
print("Lending Activity by State:")
print(state_df.head(10).to_string(index=False))


## 8. Lending Deserts

Census tracts with abnormally low application volumes combined with
high denial rates — potential indicators of redlining or lending avoidance.


In [ ]:
deserts = lending_desert_score(df)
desert_tracts = deserts[deserts["is_lending_desert"] == True]

print(f"Total census tracts analyzed: {len(deserts):,}")
print(f"Identified lending deserts:   {len(desert_tracts):,}")
print(f"\nTop 10 Lending Deserts by Score:")
print(desert_tracts[["census_tract", "applications", "denial_rate",
                       "app_percentile", "desert_score"]].head(10).to_string(index=False))


## 9. Lender Analysis

Compare a single lender's performance against the overall market.


In [ ]:
# Get the top lender by volume
top_lenders = top_lenders_by_volume(df, n=5)
print("Top 5 Lenders by Origination Volume:")
print(top_lenders.to_string(index=False))

# Analyze the top lender
top_lei = top_lenders["lei"].iloc[0]
print(f"\nAnalyzing lender: {top_lei}")
summary = lender_summary(df, lei=top_lei)
print(f"\nLender Summary:")
for k, v in summary.items():
    print(f"  {k:<25} {v}")


## 10. Lender vs Market Comparison

In [ ]:
top_lei = top_lenders_by_volume(df, n=1)["lei"].iloc[0]
comparison = lender_vs_market(df, lei=top_lei)

print(f"Lender vs Market Denial Rates — {top_lei}")
print("=" * 75)
for _, row in comparison.iterrows():
    lender_rate = row["lender_denial_rate"] * 100
    market_rate = row["market_denial_rate"] * 100
    diff = row["vs_market_pct"]
    direction = "▲ above" if diff > 0 else "▼ below"
    print(f"  {row['derived_race']:<40} "
          f"Lender: {lender_rate:5.1f}%  "
          f"Market: {market_rate:5.1f}%  "
          f"{abs(diff):.1f}% {direction} market")


## 11. Full Disparity Report

In [ ]:
report = generate_disparity_report(
    df,
    title="National Mortgage Market — 2023 HMDA Analysis"
)
print(report[:3000])
print("\n[... report continues ...]")


## 12. Real CFPB API Usage

To analyze real HMDA data from the CFPB:


In [ ]:
# Uncomment to run with real CFPB data:

# Load Illinois mortgage data for 2023
# il_df = load_from_api(year=2023, state="IL", limit=50_000)

# Load data for a specific lender (use their LEI)
# lender_df = load_from_api(year=2023, lei="BANK_LEI_HERE")

# Load data for a specific county (e.g. Cook County, IL = 17031)
# county_df = load_from_api(year=2023, county="17031")

print("CFPB HMDA Data Browser API:")
print("  URL: https://ffiec.cfpb.gov/data-browser/")
print("  Free, no API key required")
print("  2024 data: 4,908 institutions")
print()
print("Load real data with:")
print("  from hmdaanalyzer import load_from_api")
print("  df = load_from_api(year=2023, state='IL')")


## Summary

This notebook demonstrated the full hmda-analyzer workflow:

1. **Data loading** — sample data or live CFPB API
2. **Denial rates by race** — core fair lending metric
3. **Disparity ratios** — CFPB-standard comparison vs White applicants
4. **Income band analysis** — income vs race effects on denial rates
5. **Denial reasons** — what reasons are cited by race
6. **Geographic analysis** — lending patterns by state, county, tract
7. **Lending deserts** — tracts with abnormally low application volumes
8. **Lender analysis** — individual lender vs market comparison
9. **Full report** — formatted Markdown disparity analysis report

**Key finding from sample data:** Black or African American applicants
face denial rates approximately 2x higher than White applicants —
consistent with national 2023 HMDA statistics.

**GitHub:** https://github.com/Jaypatel1511/hmda-analyzer
**PyPI:** https://pypi.org/project/hmda-analyzer
**Data:** https://ffiec.cfpb.gov/data-browser/
